In [22]:
pip install requests

Note: you may need to restart the kernel to use updated packages.


In [23]:
# Install: pip install datasets
from datasets import load_dataset
import requests
from PIL import Image
import pandas as pd
from pathlib import Path
 
# Load dataset from HuggingFace
print("Loading product dataset...")
try:
    # Try loading the dataset
    dataset = load_dataset("ashraq/fashion-product-images-small", split="train[:100]")  # First 100 samples
    print(f"✓ Loaded {len(dataset)} products")
    
    # Convert to pandas for easier manipulation
    products_df = pd.DataFrame(dataset)
    print(f"Dataset columns: {products_df.columns.tolist()}")
    
except Exception as e:
    print(f"⚠ Could not load HuggingFace dataset: {e}")
    print("Using local images instead...")
    
    # Alternative: Use local images
    # Create a products.json file with product information
    products_data = [
        {
            "id": 1,
            "name": "Wireless Headphones",
            "price": 79.99,
            "category": "Electronics",
            "image_path": "images/product1.jpg"
        },
        # Add more products...
    ]
    
    products_df = pd.DataFrame(products_data)
 
# Create images directory
images_dir = Path("product_images")
images_dir.mkdir(exist_ok=True)
 
print(f"\n✓ Dataset prepared!")
print(f"  Total products: {len(products_df)}")

Loading product dataset...
✓ Loaded 100 products
Dataset columns: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image']

✓ Dataset prepared!
  Total products: 100


In [28]:
import base64

# ---- Step 3: function to encode an image ----
def encode_image_to_base64(image_path):
    """Encode an image file to base64 string."""
    with open(image_path, "rb") as img_file:
        encoded = base64.b64encode(img_file.read()).decode("utf-8")
    return encoded


# ---- Save images as .jpg files and add the missing columns ----
placeholder_prices = {
    "Apparel": 29.99,
    "Accessories": 24.99,
    "Footwear": 59.99,
    "Personal Care": 14.99,
}

image_paths = []      # with an "s"
prices = []

for index, row in products_df.iterrows():
    image_path = images_dir / f"{row['id']}.jpg"
    row["image"].convert("RGB").save(image_path)
    image_paths.append(str(image_path))
    prices.append(placeholder_prices.get(row["masterCategory"], 19.99))

products_df["image_path"] = image_paths
products_df["price"] = prices
products_df["name"] = products_df["productDisplayName"]
products_df["category"] = products_df["masterCategory"] + " - " + products_df["articleType"]

print(f"✓ Saved {len(image_paths)} images to '{images_dir}'")
display(products_df[["id", "name", "price", "category", "image_path"]].head())


# ---- Example usage ----
sample_path = products_df.iloc[0]["image_path"]
encoded_image = encode_image_to_base64(sample_path)
print(f"Encoded image length: {len(encoded_image)} characters")
print(f"Encoded prefix: {encoded_image[:40]}...")

✓ Saved 100 images to 'product_images'


,id,name,price,category,image_path
0,15970,Turtle Check Men Navy Blue Shirt,29.99,Apparel - Shirts,product_images/15970.jpg
1,39386,Peter England Men Party Blue Jeans,29.99,Apparel - Jeans,product_images/39386.jpg
2,59263,Titan Women Silver Watch,24.99,Accessories - Watches,product_images/59263.jpg
3,21379,Manchester United Men Solid Black Track Pants,29.99,Apparel - Track Pants,product_images/21379.jpg
4,53759,Puma Men Grey T-shirt,29.99,Apparel - Tshirts,product_images/53759.jpg


Encoded image length: 2388 characters
Encoded prefix: /9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcG...


In [29]:
def create_product_listing_prompt(product_name, price, category, additional_info=None):
    """
    Create a prompt for generating product listings.
    
    Parameters:
    - product_name: Name of the product
    - price: Price of the product
    - category: Product category
    - additional_info: Optional additional information
    
    Returns:
    - Formatted prompt string
    """
    prompt = f"""You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.
 
Product Information:
- Name: {product_name}
- Price: ${price:.2f}
- Category: {category}
{f'- Additional Info: {additional_info}' if additional_info else ''}
 
Please create a professional product listing that includes:
 
1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive language
   - Include relevant details visible in the image
3. **Key Features** (bullet points, 5-7 items)
4. **SEO Keywords** (comma-separated, 10-15 relevant keywords)
 
Format your response as JSON with the following structure:
{{
    "title": "Product title here",
    "description": "Full description here",
    "features": ["Feature 1", "Feature 2", ...],
    "keywords": "keyword1, keyword2, ..."
}}
 
Be specific about what you see in the image. Mention colors, materials, design elements, and any distinctive features."""
    
    return prompt
 
# Test prompt creation
test_prompt = create_product_listing_prompt(
    product_name="Wireless Bluetooth Headphones",
    price=79.99,
    category="Electronics",
    additional_info="Noise cancelling, 30-hour battery"
)
 
print("\n" + "="*50)
print("PROMPT TEMPLATE")
print("="*50)
print(test_prompt[:500] + "...")  # Show first 500 characters


PROMPT TEMPLATE
You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.

Product Information:
- Name: Wireless Bluetooth Headphones
- Price: $79.99
- Category: Electronics
- Additional Info: Noise cancelling, 30-hour battery

Please create a professional product listing that includes:

1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive ...


In [30]:
import random
if "price" not in products_df.columns:
    products_df["price"] = [round(random.uniform(15, 120), 2) for _ in range(len(products_df))]

In [31]:
import os
import json
import time
from openai import OpenAI, RateLimitError, APIError, APIConnectionError

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))  # reuse your Step 1 client if you already have one

MODEL = "gpt-4o-mini"  # cheap and vision-capable; swap in whatever model Ironhack tells you to use


def generate_listing(row, max_retries=3):
    """Send one product's image + metadata to the API and return the parsed listing as a dict."""
    # 1. Build the prompt from the row's metadata
    extra = f"Colour: {row['baseColour']}, Gender: {row['gender']}, Usage: {row['usage']}, Season: {row['season']}"
    prompt = create_product_listing_prompt(
        product_name=row["productDisplayName"],
        price=row["price"],
        category=row["articleType"],
        additional_info=extra,
    )

    # 2. Encode the image
    image_b64 = encode_image_to_base64(row["image_path"])

    # 3. Call the API, retrying on temporary errors
    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[{
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {"type": "image_url",
                         "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"}},
                    ],
                }],
                response_format={"type": "json_object"},  # forces valid JSON back
                max_tokens=800,
                temperature=0.7,
            )
            content = response.choices[0].message.content
            return json.loads(content)  # 4. Parse JSON into a Python dict

        except RateLimitError:
            wait = 2 ** attempt
            print(f"  Rate limited, waiting {wait}s (attempt {attempt}/{max_retries})")
            time.sleep(wait)
        except (APIConnectionError, APIError) as e:
            print(f"  API error: {e} (attempt {attempt}/{max_retries})")
            time.sleep(2)
        except json.JSONDecodeError:
            print("  Response wasn't valid JSON, retrying...")

    raise RuntimeError(f"Failed after {max_retries} attempts")


# Test on the first product
test_row = products_df.iloc[0]
listing = generate_listing(test_row)

print("Product:", test_row["productDisplayName"])
print(json.dumps(listing, indent=2))

Product: Turtle Check Men Navy Blue Shirt
{
  "title": "Stylish Turtle Check Men\u2019s Navy Blue Shirt - Fall Essential",
  "description": "Elevate your casual wardrobe with the Turtle Check Men\u2019s Navy Blue Shirt, designed for comfort and style. This shirt features a classic check pattern that exudes a timeless charm, making it perfect for any relaxed outing this fall. Crafted from high-quality materials, it ensures durability while providing a soft, breathable feel against your skin. The rich navy blue color is versatile, allowing you to pair it effortlessly with jeans or chinos for a polished yet laid-back look. With its short sleeves and lightweight fabric, this shirt is ideal for layering under a jacket or wearing solo. Whether you're heading out for brunch or enjoying a casual day out, the Turtle Check shirt will keep you looking sharp and stylish all season long.",
  "features": [
    "Classic navy blue color perfect for fall",
    "Timeless check pattern for a stylish look

In [32]:
N_PRODUCTS = 10  # keep it small while testing; each call costs money

results = []
errors = []

for i, row in products_df.head(N_PRODUCTS).iterrows():
    print(f"[{len(results) + len(errors) + 1}/{N_PRODUCTS}] {row['productDisplayName']}")
    try:
        listing = generate_listing(row)
        results.append({
            "id": int(row["id"]),
            "original_name": row["productDisplayName"],
            "category": row["articleType"],
            "price": float(row["price"]),
            "image_path": row["image_path"],
            "title": listing.get("title"),
            "description": listing.get("description"),
            "features": listing.get("features", []),
            "keywords": listing.get("keywords"),
        })
        print("  ✓ done")
    except Exception as e:
        errors.append({"id": int(row["id"]), "error": str(e)})
        print(f"  ✗ failed: {e}")
    time.sleep(1)  # be polite to the API

# Save results
with open("product_listings.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

listings_df = pd.DataFrame(results)
listings_df["features"] = listings_df["features"].apply(lambda x: " | ".join(x))  # CSV-friendly
listings_df.to_csv("product_listings.csv", index=False)

print(f"\n✓ {len(results)} succeeded, ✗ {len(errors)} failed")
print("Saved to product_listings.json and product_listings.csv")
listings_df[["original_name", "title"]].head()

[1/10] Turtle Check Men Navy Blue Shirt
  ✓ done
[2/10] Peter England Men Party Blue Jeans
  ✓ done
[3/10] Titan Women Silver Watch
  ✓ done
[4/10] Manchester United Men Solid Black Track Pants
  ✓ done
[5/10] Puma Men Grey T-shirt
  ✓ done
[6/10] Inkfruit Mens Chain Reaction T-shirt
  ✓ done
[7/10] Fabindia Men Striped Green Shirt
  ✓ done
[8/10] Jealous 21 Women Purple Shirt
  ✓ done
[9/10] Puma Men Pack of 3 Socks
  ✓ done
[10/10] Skagen Men Black Watch
  ✓ done

✓ 10 succeeded, ✗ 0 failed
Saved to product_listings.json and product_listings.csv


,original_name,title
0,Turtle Check Men Navy Blue Shirt,Stylish Turtle Check Navy Blue Shirt for Men
1,Peter England Men Party Blue Jeans,Stylish Peter England Blue Jeans for Men - Sum...
2,Titan Women Silver Watch,Elegant Titan Women Silver Watch - Perfect for...
3,Manchester United Men Solid Black Track Pants,Manchester United Men's Solid Black Track Pants
4,Puma Men Grey T-shirt,Stylish Puma Men's Grey T-Shirt for Casual Sum...
